# ExoTwin: Digital Twin for Exoplanet Habitability

**ENHANCE HACK-4-SAGES 2026** | Category C: Digital Twins in Exoplanet Habitability

---

## Abstract

With over 5,700 confirmed exoplanets discovered to date, a key question in astrobiology is: which of these worlds could harbour microbial life? We present **ExoTwin**, a digital twin framework that integrates real observational data from two authoritative sources—the NASA Exoplanet Archive (6138 planets) and the PHL Habitable Worlds Catalog (expert-curated habitability labels for 70 candidates)—with physics-based models and machine learning to predict the habitability of exoplanets. Our approach computes a multi-dimensional habitability profile for each planet—including habitable zone membership, Earth Similarity Index, atmosphere retention capability, and surface conditions—then trains an ML model to predict a composite habitability score. Cross-validation against PHL expert labels and Solar System benchmarks confirms model reliability. The interactive digital twin dashboard allows users to explore "what-if" scenarios by modifying planetary parameters in real time, enabling researchers and educators to investigate which factors most influence habitability.

---

## 1. Introduction

### 1.1 Research Question

> **Given observable parameters of an exoplanet, can a digital twin model predict the probability of conditions suitable for microbial life, and identify which factors contribute most to habitability?**

### 1.2 Background

The search for life beyond Earth is one of the most profound scientific endeavours of our time. Current and future missions—JWST, the Habitable Worlds Observatory (HWO), and the LIFE telescope—are generating unprecedented data on exoplanet atmospheres and compositions. However, with thousands of known exoplanets, prioritising observation targets requires a systematic assessment of habitability.

Habitability depends on multiple factors: orbital distance from the host star (habitable zone), planetary mass and radius (rocky composition), the ability to retain an atmosphere (escape velocity), surface temperature, stellar activity, and orbital stability. Traditional approaches compute individual metrics (e.g., the Earth Similarity Index by Schulze-Makuch et al., 2011, or habitable zone boundaries by Kopparapu et al., 2013), but rarely combine them into a unified predictive framework.

### 1.3 What Is a Digital Twin?

A digital twin is a virtual representation of a physical entity that:
1. **Integrates real observational data** from telescopes and surveys
2. **Models physical processes** using established astrophysical equations
3. **Predicts future states** using machine learning
4. **Enables exploration** through interactive "what-if" simulation
5. **Updates dynamically** as new data becomes available

ExoTwin applies this concept to exoplanets: each planet in the NASA archive becomes a virtual twin that can be inspected, modified, and compared.

---

## 2. Data Sources

| Source | Description | What it provides | URL |
|--------|-------------|------------------|-----|
| NASA Exoplanet Archive | Primary dataset, 6138 confirmed exoplanets | Observed planetary and stellar parameters (mass, radius, orbit, temperature, etc.) | https://exoplanetarchive.ipac.caltech.edu/ |
| PHL Habitable Worlds Catalog | 5599 planets with expert analysis | Expert-curated habitability labels (`P_HABITABLE`: 0/1/2), independently computed ESI for 5358 planets, planet type classification, habitable zone flags | https://phl.upr.edu/hwc |
| Solar System benchmarks | Earth, Mars, Venus for validation | Known ground-truth for model sanity checks | Standard astronomical data |

### 2.1 PHL Integration

The PHL Habitable Worlds Catalog (Méndez et al., UPR Arecibo) provides two key additions:

1. **Expert habitability labels** — 70 planets tagged as potentially habitable (29 conservative, 41 optimistic) by planetary scientists using criteria from Kopparapu et al. (2014). These serve as independent validation for our model predictions.
2. **ESI gap-filling** — PHL computes ESI for 5358 planets. By merging this with our dataset, we increased ESI coverage from 1211 to 5582 planets. Our independently computed ESI correlates **r = 0.89** with PHL's values (MAE = 0.052), confirming the correctness of our feature engineering.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='darkgrid', palette='viridis')

df = pd.read_csv('../data/processed/exoplanets_final.csv')
print(f'Dataset: {len(df)} exoplanets, {len(df.columns)} columns')
print(f'PHL matches: {df["phl_esi"].notna().sum()} planets with PHL data')
print(f'PHL habitable: {(df["phl_habitable"]>0).sum()} expert-tagged candidates')
df.head()

## 3. Feature Description

### 3.1 Observed Parameters (from NASA)

| Parameter | Description | Unit |
|-----------|-------------|------|
| `pl_bmasse` | Planet mass | Earth masses |
| `pl_rade` | Planet radius | Earth radii |
| `pl_orbper` | Orbital period | days |
| `pl_orbsmax` | Semi-major axis | AU |
| `pl_orbeccen` | Orbital eccentricity | — |
| `pl_eqt` | Equilibrium temperature | K |
| `pl_dens` | Bulk density | g/cm³ |
| `st_teff` | Stellar effective temperature | K |
| `st_lum` | Stellar luminosity | log₁₀(Solar) |
| `st_mass` | Stellar mass | Solar masses |
| `st_rad` | Stellar radius | Solar radii |

### 3.2 Engineered Features

| Feature | Formula / Source | Significance |
|---------|------------------|-------------|
| `escape_velocity` | √(2GM/R) | Can the planet retain an atmosphere? |
| `stellar_flux` | L / a² | Energy received from the star |
| `hz_inner`, `hz_outer` | Kopparapu et al. (2013) | Habitable zone boundaries |
| `in_hz` | inner ≤ a ≤ outer | Is it in the habitable zone? |
| `is_rocky` | ρ > 3.5 g/cm³ or R < 1.6 R⊕ | Rocky vs. gaseous |
| `esi` | Schulze-Makuch et al. (2011) | Earth Similarity Index (0–1) |
| `habitability_score` | Weighted composite | Target variable (0–1) |

### 3.3 Planet Classification

Exoplanets span a wide range of sizes and compositions. Their classification is relevant to habitability assessment:

| Category | Criteria | Habitability relevance |
|----------|----------|----------------------|
| **Rocky (Terran)** | R < 1.6 R⊕ or ρ > 3.5 g/cm³ | Most likely to support surface life |
| **Super-Earths** | 1.2–2.0 R⊕, 2–10 M⊕ | May be rocky with thicker atmospheres |
| **Mini-Neptunes** | 2.0–4.0 R⊕ | Thick H/He envelopes reduce surface habitability |
| **Gas Giants** | R > 4 R⊕, M > 50 M⊕ | No solid surface — not habitable (moons may be) |

### 3.4 Tidal Locking

Many habitable zone candidates around M-dwarf stars (T_eff < 3500 K) are **tidally locked** — one hemisphere permanently faces the star. This includes TRAPPIST-1 e/f/g, Proxima Centauri b, and Teegarden's Star b, all with orbital periods under 15 days and semi-major axes below 0.05 AU.

Tidal locking is not directly flagged in NASA or PHL datasets, but it has significant implications for habitability:
- The **dayside** may be too hot, the **nightside** too cold
- The **terminator zone** (day-night boundary) could sustain liquid water
- **Atmospheric circulation** patterns determine whether heat is redistributed

This is listed as a limitation of our model — tidal locking affects habitability but cannot be directly measured from current transit/radial velocity observations.

## 4. Exploratory Data Analysis

In [ ]:
# 4.1 Missing data overview
key_cols = ['pl_bmasse','pl_rade','pl_orbper','pl_orbsmax','pl_orbeccen','pl_eqt',
            'st_teff','st_lum','st_mass','st_rad','escape_velocity','stellar_flux',
            'in_hz','is_rocky','esi','habitability_score']

missing_pct = (df[key_cols].isnull().sum() / len(df) * 100).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(10, 6))
missing_pct.plot.barh(ax=ax, color=plt.cm.viridis(missing_pct.values / 100))
ax.set_xlabel('Missing (%)')
ax.set_title('Missing Data by Feature')
for i, (v, name) in enumerate(zip(missing_pct.values, missing_pct.index)):
    ax.text(v + 0.5, i, f'{v:.1f}%', va='center', fontsize=9)
plt.tight_layout()
plt.savefig('../assets/missing_data.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# 4.2 Habitability score distribution
hs = df['habitability_score'].dropna()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(hs, bins=50, color='#2ecc71', edgecolor='black', alpha=0.8)
axes[0].set_xlabel('Habitability Score')
axes[0].set_ylabel('Number of Planets')
axes[0].set_title('Distribution of Habitability Scores')
axes[0].axvline(0.5, color='red', linestyle='--', label='Threshold 0.5')
axes[0].legend()

axes[1].hist(hs[hs > 0.2], bins=30, color='#e74c3c', edgecolor='black', alpha=0.8)
axes[1].set_xlabel('Habitability Score')
axes[1].set_ylabel('Number of Planets')
axes[1].set_title('Zoom: Planets with Score > 0.2')

plt.tight_layout()
plt.savefig('../assets/habitability_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Planets with score > 0.5: {(hs > 0.5).sum()}')
print(f'Planets with score > 0.3: {(hs > 0.3).sum()}')

In [ ]:
# 4.3 Mass vs Radius (planet type diagram)
subset = df.dropna(subset=['pl_bmasse', 'pl_rade', 'habitability_score'])

fig = px.scatter(
    subset,
    x='pl_bmasse', y='pl_rade',
    color='habitability_score',
    color_continuous_scale='RdYlGn',
    hover_name='pl_name',
    hover_data=['pl_eqt', 'in_hz', 'esi', 'is_rocky'],
    log_x=True, log_y=True,
    labels={'pl_bmasse': 'Planet Mass (Earth masses)', 'pl_rade': 'Planet Radius (Earth radii)',
            'habitability_score': 'Habitability'},
    title='Exoplanet Mass vs Radius (colored by Habitability Score)',
    width=900, height=600,
)
fig.add_scatter(x=[1], y=[1], mode='markers', marker=dict(size=15, color='blue', symbol='star'),
                name='Earth', showlegend=True)
fig.show()

In [ ]:
# 4.4 Habitable Zone diagram
hz_data = df.dropna(subset=['pl_orbsmax', 'st_teff', 'hz_inner', 'hz_outer'])

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=hz_data['st_teff'], y=hz_data['hz_inner'],
    mode='markers', marker=dict(size=2, color='orange', opacity=0.3),
    name='HZ Inner Edge'
))
fig.add_trace(go.Scatter(
    x=hz_data['st_teff'], y=hz_data['hz_outer'],
    mode='markers', marker=dict(size=2, color='cyan', opacity=0.3),
    name='HZ Outer Edge'
))

in_hz = hz_data[hz_data['in_hz'] == 1]
out_hz = hz_data[hz_data['in_hz'] == 0]

fig.add_trace(go.Scatter(
    x=out_hz['st_teff'], y=out_hz['pl_orbsmax'],
    mode='markers', marker=dict(size=4, color='gray', opacity=0.2),
    name='Outside HZ', text=out_hz['pl_name']
))
fig.add_trace(go.Scatter(
    x=in_hz['st_teff'], y=in_hz['pl_orbsmax'],
    mode='markers', marker=dict(size=8, color='green', opacity=0.8),
    name='Inside HZ', text=in_hz['pl_name']
))

fig.update_layout(
    title='Habitable Zone Diagram: Stellar Temperature vs Orbital Distance',
    xaxis_title='Stellar Effective Temperature (K)',
    yaxis_title='Semi-major Axis (AU)',
    yaxis_type='log',
    width=900, height=600,
)
fig.show()

In [ ]:
# 4.5 Correlation heatmap of key features
corr_cols = ['pl_bmasse','pl_rade','pl_orbsmax','pl_eqt','pl_dens',
             'st_teff','st_lum','escape_velocity','stellar_flux',
             'esi','habitability_score']
corr = df[corr_cols].corr()

fig, ax = plt.subplots(figsize=(12, 10))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, square=True, linewidths=0.5, ax=ax)
ax.set_title('Feature Correlation Matrix')
plt.tight_layout()
plt.savefig('../assets/correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# 4.6 Top habitable candidates
top20 = df.nlargest(20, 'habitability_score')[
    ['pl_name','habitability_score','esi','in_hz','pl_eqt','pl_rade','pl_bmasse','is_rocky','escape_velocity']
].reset_index(drop=True)
top20.index += 1
top20.style.background_gradient(subset=['habitability_score','esi'], cmap='RdYlGn')

In [ ]:
# 4.7 Discovery method breakdown
methods = df['discoverymethod'].value_counts()

fig = px.pie(values=methods.values, names=methods.index,
             title='Exoplanet Discovery Methods',
             color_discrete_sequence=px.colors.qualitative.Set2)
fig.update_traces(textposition='inside', textinfo='percent+label')
fig.update_layout(width=700, height=500)
fig.show()

print('Detection bias: Transit method (74%) favors large planets close to their star.')
print('This means our dataset under-represents small, far-out rocky planets in habitable zones.')

## 5. Model Training

### 5.1 Methodology

We trained two tree-based regression models to predict the composite `habitability_score` (0–1) from 16 input features (10 observed + 6 engineered). Missing values were imputed with column medians. Both models were tuned via 5-fold cross-validated grid search.

| Model | Hyperparameter Grid |
|-------|-------------------|
| Random Forest | n_estimators: [100, 200, 300], max_depth: [10, 15, 20, None], min_samples_split: [2, 5] |
| Gradient Boosting | n_estimators: [100, 200, 300], max_depth: [3, 5, 7], learning_rate: [0.05, 0.1, 0.2] |

### 5.2 Mentor Input

During our Day 1 mentor session, we were advised to cross-reference our data with the **PHL Habitable Worlds Catalog** (Méndez et al., UPR Arecibo). This was integrated on Day 2, providing expert-curated habitability labels for 70 planets and ESI values for 5358 planets — used both for ESI gap-filling and as an independent validation set.

In [ ]:
# 5.3 Training results
import json, joblib

with open('../models/metadata.json') as f:
    meta = json.load(f)

results = pd.DataFrame(meta['metrics'])
results = results[['model','r2','mse','mae','cv_r2_mean']]
results.columns = ['Model', 'R²', 'MSE', 'MAE', 'CV R² (mean)']
results = results.round(6)

print(f"Best model: {meta['model_name']}")
print(f"Features used: {len(meta['features'])}")
print(f"Target: {meta['target']}")
print()
results.style.highlight_max(subset=['R²', 'CV R² (mean)'], color='#d5f5e3') \
             .highlight_min(subset=['MSE', 'MAE'], color='#d5f5e3')

## 6. Explainability (SHAP)

SHAP (SHapley Additive exPlanations) values reveal which features drive the model's predictions — not just *what* the model predicts, but *why*. We use `shap.TreeExplainer` on the Gradient Boosting model.

In [ ]:
# 6.1 SHAP Feature Importance
from IPython.display import Image, display

display(Image(filename='../assets/shap_importance.png', width=800))
print("Top features by mean |SHAP| value:")
print("  1. is_rocky         — rocky composition is the strongest habitability signal")
print("  2. escape_velocity  — ability to retain an atmosphere")
print("  3. esi              — overall Earth similarity")
print("  4. pl_dens          — bulk density (rocky vs gaseous)")
print("  5. pl_eqt           — equilibrium temperature")
print("  6. in_hz            — habitable zone membership")

In [ ]:
# 6.2 SHAP Summary Plot (beeswarm)
display(Image(filename='../assets/shap_summary.png', width=800))
print("Each dot is one planet. Red = high feature value, blue = low.")
print("Features pushing predictions right (higher score) increase habitability.")

## 7. Validation

Model validation is performed at two levels:
1. **Solar System benchmarks** — planets with known properties (Earth, Mars, Venus, Jupiter) serve as sanity checks.
2. **PHL expert labels** — 70 planets tagged as potentially habitable by planetary scientists at PHL (UPR Arecibo) provide independent validation.

### 7.1 Solar System + Key Exoplanet Benchmarks

In [ ]:
# 7.1 Validation on known planets
val = pd.read_csv('../assets/validation_results.csv')
val['Predicted Score'] = val['Predicted Score'].round(3)
val.columns = ['Planet', 'Predicted Score', 'Expected Range']

fig = go.Figure(go.Bar(
    x=val['Predicted Score'], y=val['Planet'],
    orientation='h',
    marker_color=['#3fb950' if s > 0.7 else '#58a6ff' if s > 0.4 else '#8b949e'
                  for s in val['Predicted Score']],
    text=val['Predicted Score'].apply(lambda x: f'{x:.3f}'),
    textposition='outside',
))
fig.update_layout(
    title='Model Validation: Predicted Habitability of Known Planets',
    xaxis=dict(title='Predicted Habitability Score', range=[0, 1.05]),
    yaxis=dict(autorange='reversed'),
    height=400, width=800,
    margin=dict(l=120, r=40, t=50, b=40),
)
fig.show()

print("Interpretation:")
print("  Earth (0.952): Highest score — correct, it is the only confirmed habitable planet")
print("  TRAPPIST-1 e (0.922): Rocky planet in habitable zone of TRAPPIST-1 — top candidate in literature")
print("  Proxima Cen b (0.912): Nearest exoplanet, in HZ — consistent with scientific consensus")
print("  Kepler-442 b (0.885): Well-known HZ super-Earth — expected high score")
print("  K2-18 b (0.648): Sub-Neptune, JWST detected CO2/CH4 — uncertain habitability, medium score appropriate")
print("  Venus (0.637): Similar size to Earth but runaway greenhouse — model gives moderate score")
print("  Mars (0.594): Past liquid water, thin atmosphere — medium score reasonable")
print("  Jupiter (0.227): Gas giant, no surface — correctly low")

### 7.2 PHL Expert Label Validation

The PHL Habitable Worlds Catalog independently labels 70 planets as potentially habitable: 29 in the "conservative" sample (rocky, in habitable zone) and 41 in the "optimistic" sample (may include water worlds or mini-Neptunes). We check if our model's predicted scores align with these expert assessments.

In [ ]:
# 7.2 PHL expert label validation
model = joblib.load('../models/best_model.pkl')
imputer = joblib.load('../models/imputer.pkl')

FEATURE_COLS = meta['features']

has_label = df[df['phl_habitable'].notna()].copy()
X_lab = has_label[FEATURE_COLS]
X_lab_imp = pd.DataFrame(imputer.transform(X_lab), columns=FEATURE_COLS, index=X_lab.index)
has_label['predicted'] = model.predict(X_lab_imp)

groups = {0: 'Not habitable', 1: 'Conservative', 2: 'Optimistic'}
summary = []
for label, desc in groups.items():
    subset = has_label[has_label['phl_habitable'] == label]['predicted']
    summary.append({'PHL Label': desc, 'Count': len(subset),
                    'Mean Score': subset.mean(), 'Median': subset.median(),
                    'Min': subset.min(), 'Max': subset.max()})
summary_df = pd.DataFrame(summary).round(4)
display(summary_df)

# Box plot
fig = go.Figure()
for label, desc, color in [(0, 'Not habitable', '#8b949e'), (1, 'Conservative', '#3fb950'), (2, 'Optimistic', '#58a6ff')]:
    scores = has_label[has_label['phl_habitable'] == label]['predicted']
    fig.add_trace(go.Box(y=scores, name=desc, marker_color=color, boxmean=True))
fig.update_layout(
    title='Model Scores by PHL Habitability Label',
    yaxis_title='Predicted Habitability Score',
    height=450, width=700,
)
fig.show()

hab = has_label[has_label['phl_habitable'] > 0]['predicted']
non = has_label[has_label['phl_habitable'] == 0]['predicted']
print(f"Mean score separation (habitable - non-habitable): {hab.mean() - non.mean():.4f}")
print(f"PHL habitable planets scoring > 0.3: {(hab > 0.3).sum()}/{len(hab)} ({100*(hab>0.3).sum()/len(hab):.0f}%)")

### 7.3 ESI Cross-Validation (Ours vs PHL)

In [ ]:
# 7.3 ESI cross-validation: our computed ESI vs PHL's independent ESI
both = df.dropna(subset=['esi', 'phl_esi']).copy()
# Only compare where we computed ESI ourselves (not gap-filled from PHL)
# We need the original features dataset for this
original = pd.read_csv('../data/processed/exoplanets_features.csv')
original_esi = original[['pl_name', 'esi']].rename(columns={'esi': 'our_esi_original'})
both = both.merge(original_esi, on='pl_name', how='left')
both_valid = both.dropna(subset=['our_esi_original', 'phl_esi'])

if len(both_valid) > 10:
    corr = both_valid['our_esi_original'].corr(both_valid['phl_esi'])
    mae = (both_valid['our_esi_original'] - both_valid['phl_esi']).abs().mean()

    fig = px.scatter(both_valid, x='phl_esi', y='our_esi_original',
                     hover_name='pl_name', opacity=0.5,
                     labels={'phl_esi': 'PHL ESI', 'our_esi_original': 'Our Computed ESI'},
                     title=f'ESI Cross-Validation: Ours vs PHL (r = {corr:.3f}, MAE = {mae:.3f})',
                     width=700, height=500)
    fig.add_scatter(x=[0, 1], y=[0, 1], mode='lines', line=dict(dash='dash', color='gray'),
                    name='Perfect agreement', showlegend=True)
    fig.show()

    print(f"Compared {len(both_valid)} planets where we independently computed ESI")
    print(f"Pearson correlation: r = {corr:.4f}")
    print(f"Mean absolute error: {mae:.4f}")
    print("This confirms our feature engineering is correct and consistent with published methods.")
else:
    print(f"Only {len(both_valid)} planets with both ESI values — not enough for comparison.")

## 8. Discussion

### 8.1 Key Findings

1. **Rocky composition is the strongest predictor of habitability.** The SHAP analysis reveals that `is_rocky` and `escape_velocity` (atmosphere retention) dominate over orbital parameters. This aligns with the astrobiological consensus that a solid surface and atmosphere are prerequisites for life as we know it.

2. **The model generalises well to known benchmarks.** Earth receives the highest score (0.952), gas giants score low (Jupiter: 0.227), and well-studied candidates like TRAPPIST-1 e (0.922) and Proxima Centauri b (0.912) score high — consistent with published assessments.

3. **PHL expert labels provide independent validation.** Conservative habitable planets score 3.5× higher than non-habitable planets on average (mean 0.60 vs 0.13), confirming that our model captures meaningful habitability signals.

4. **ESI cross-validation confirms feature engineering correctness.** Our independently computed ESI correlates r = 0.89 with PHL's values, demonstrating that our physics-based computations (Schulze-Makuch et al. 2011) are correctly implemented.

### 8.2 Limitations

- **Detection bias:** The Transit method (74% of discoveries) favors large planets close to their stars. Our training set under-represents small, rocky planets in wide habitable zones — precisely the most interesting targets.
- **No atmospheric data:** For most exoplanets, atmospheric composition is unknown. Habitability depends critically on whether a greenhouse atmosphere exists, but our model cannot account for this.
- **Earth-centric definition:** Our habitability score is calibrated to Earth-like conditions (liquid water, moderate temperature, rocky surface). Alternative biochemistries or subsurface ocean worlds (e.g., Europa, Enceladus) may host life under very different conditions.
- **Synthetic target variable:** Our `habitability_score` is derived from physical models, not from observed evidence of life. The model learns to reproduce this composite score, not to detect actual biosignatures.
- **Missing data imputation:** Median imputation for missing features introduces noise. Planets with more missing values have less reliable predictions.

### 8.3 Future Work

- **JWST atmospheric integration:** As JWST delivers atmospheric spectra for more exoplanets (e.g., CO₂, CH₄, O₃ detection on K2-18 b), these can be incorporated as additional features.
- **Bayesian uncertainty:** Replace point estimates with posterior distributions to quantify prediction confidence.
- **Subsurface habitability:** Extend the model to assess subsurface ocean worlds (tidal heating, ice shell thickness) following Affholder et al. (2021).
- **Real-time data updates:** Connect the digital twin to NASA's TAP API for automatic updates as new exoplanets are confirmed.

## 9. References

1. Kopparapu, R. K. et al. (2013). "Habitable Zones around Main-Sequence Stars: New Estimates." *ApJ*, 765(2), 131.
2. Schulze-Makuch, D. et al. (2011). "A Two-Tiered Approach to Assessing the Habitability of Exoplanets." *Astrobiology*, 11(10), 1041–1052.
3. Affholder, A. et al. (2021). "Bayesian analysis of Enceladus's plume data to assess methanogenesis." *Nature Astronomy*, 5, 805–814.
4. Meadows, V. S. & Barnes, R. K. (2018). "Factors Affecting Exoplanet Habitability." In *Handbook of Exoplanets*.
5. Méndez, A. et al. (2021). "Habitable Worlds Catalog." Planetary Habitability Laboratory, UPR Arecibo. https://phl.upr.edu/hwc
6. Madhusudhan, N. et al. (2023). "Carbon-bearing molecules in a possible hycean atmosphere." *ApJ Letters*, 956, L13.
7. NASA Exoplanet Archive. https://exoplanetarchive.ipac.caltech.edu/